In [ ]:
### This script is used to process the 2p data and corresponding treadmill behavior data and save the processed data -- before SMI calculation
### JSY, 04/2025

### This script is used to analyze the 2p data and corresponding treadmill behavior data
### 1. Load the 2p data and treadmill behavior data
### 2. Align the 2p data and treadmill behavior data
### 3a. Find temporal offset, which yields the best alignment between 2p and behavior data
### 3b. Smooth the deconvolved traces using a 250 ms Gaussian window
### 3c. Remove inactive data points
### 4. Spatial discretization (divide the VR corridor into ~110 bins, each representing 1cm and assign each data point to its corresponding spatial bin)
### 5. Test for reliability for individual cells (calculate Pearson CC or cohen’s D)
### 6. Response Plot - plotting activity of all responsive cells (cross validation – split trials in half)

In [ ]:
%load_ext autoreload
%autoreload 2
# Reloads all imported modules when they change (this is a comment on its own line)

In [ ]:
import os
import re
import datetime
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.ndimage import gaussian_filter1d

%cd "C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation"
from helper import TwoP, read_xml, time2float
from helper import SpikeSmoothing, BehavioralDataFiltering as DF, spatial_discretization as SD, ReliabilityTesting as RT, ResponseVisualization as RV, SpatialModulationIndex as SMI
from helper.SpatialModulationIndexLayerSpecific import SpatialModulationIndexLayerSpecific as SMI_Layer
from helper.detrendAdaptation import detrendAdaptation as DA

**1. Load the 2p data and treadmill behavior data**

In [ ]:
# File path to VRlog.txt and 2p data
# VRfilepath = r"D:\V1_SpatialModulation\V1_SpatialMod_VRLog"

# JSY038, BBBB
filepath = r'D:\V1_SpatialModulation\2p\250428_JSY_JSY038_SpatialModulation_BBBB\TSeries-04282025-1500-001'
twoP_filename = "TSeries-04282025-1500-001"
VRlog_filename = "VRlog_JSY038_04282025_03-27-04.txt"

# # JSY040, BBBB
# filepath = r'D:\V1_SpatialModulation\2p\250427_JSY_JSY040_SpatialModulation_BBBB\TSeries-04272025-1508-001'
# twoP_filename = "TSeries-04272025-1508-001"
# VRlog_filename = "VRlog_JSY038_04272025_03-38-36.txt"

# # JSY040, BBBB
# filepath = r'D:\V1_SpatialModulation\2p\250501_JSY_JSY040_SpatialModulation_BBBB\TSeries-05012025-1316-002'
# twoP_filename = "TSeries-05012025-1316-002"
# VRlog_filename = "VRlog_JSY038_05012025_02-18-59.txt"

# # JSY041, light intensity graudally changing, coupled movement ++ 3x landmarks, 1st recording
# filepath = r'D:\V1_SpatialModulation\2p\250415_JSY_JSY041_spatialmodulation_3xlandmarks\TSeries-04152025-1559-002'
# twoP_filename = "TSeries-04152025-1559-002"
# VRlog_filename = "VRlog_JSY03

# # JSY041, day 1
# filepath = r'D:\V1_SpatialModulation\2p\250320_JSY_JSY041_spatialmodulation\TSeries-03202025-1804-002'
# twoP_filename = "TSeries-03202025-1804-002"
# VRlog_filename = "VRlog_JSY038_03202025_06-56-38.txt"

# # JSY041, day 3
# filepath = r'D:\V1_SpatialModulation\2p\250322_JSY_JSY041_SpatialModulation\TSeries-03222025-1742-002'
# twoP_filename = "TSeries-03222025-1742-002"
# VRlog_filename = "VRlog_JSY038_03222025_06-00-15.txt"

# # # JSY041, day 5
# filepath = r'D:\V1_SpatialModulation\2p\250324_JSY_JSY041_spatialmodulation\TSeries-03242025-1618-002'
# twoP_filename = "TSeries-03242025-1618-002"
# VRlog_filename = "VRlog_JSY038_03242025_05-05-17.txt"

VRfilepath = r"D:\V1_SpatialModulation\V1_SpatialMod_VRLog"

VRlog_path = os.path.join(VRfilepath, VRlog_filename)

# Extract animal ID and date from the VR_log_filename
match = re.match(r"VRlog_(JSY\d+)_(\d{8})_\d{2}-\d{2}-\d{2}\.txt", VRlog_filename)
if match:
    animal_id = match.group(1)
    date = match.group(2)
else:
    print("Filename format does not match the expected pattern.")

# Initialize dictionaries to store raw data
twoP_data = {}
VR_data = {}

# Load twoP data
raw_twop_data = TwoP(filepath, twoP_filename)

raw_twop_data.find_files()
twop_dict = raw_twop_data.calc_dFF()

twoP_data['sps'] = twop_dict['spikes_per_sec'].copy()
twoP_data['s2p_spks'] = twop_dict['s2p_spks'].copy()
twoP_data['dFF'] = twop_dict['norm_dFF'].copy()
twoP_data['stat'] = twop_dict['stat'].copy()
twoP_data['ops'] = twop_dict['ops'].copy()

numFrames = np.size(twoP_data['sps'], 1)
numCells = len(twoP_data['stat'])

numFrames = np.size(twoP_data['sps'], 1)

xml_path = os.path.join(filepath, f"{twoP_filename}.xml")
xml_dict = read_xml(xml_path)
t0 = xml_dict["t0"]
abs_time = xml_dict["abs_time"]
rel_time = xml_dict["rel_time"]
framerate = 1/rel_time[1]
print(framerate)

twopT = np.zeros(np.size(abs_time, 0) - 1, dtype=datetime.datetime)
for rep, t in enumerate(abs_time[:-1]):
    twopT[rep] = t0 + datetime.timedelta(seconds=t)

twopT_float = time2float(twopT)
twoP_data['AbsoluteT'] = twopT

im = np.zeros((twoP_data['ops']['Ly'], twoP_data['ops']['Lx']))  # Create an empty image
for n in range(0, numCells):
    ypix = twoP_data['stat'][n]['ypix'][~twoP_data['stat'][n]['overlap']]
    xpix = twoP_data['stat'][n]['xpix'][~twoP_data['stat'][n]['overlap']]
    im[ypix, xpix] = xpix  # Assign xpix values to im for progressive color change along x-axis

# for animal facing 2p computer, image should be rotated so it goes from layer 2/3 to layer 6 (top-bottom)
# for animal facing VR computer, raw image does go from layer 2/3 to layer 6 (top-bottom)
im_rotated = np.rot90(im, k=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.imshow(im)
ax1.set_title("raw_image")

ax2.imshow(im_rotated)
ax2.set_title("rotated_image")
plt.show()

# Load VRlog
rawVR_data = []
with open(VRlog_path, "r") as file:
    lines = file.readlines()
    for line in lines[3:]:
        rawVR_data.append(line.strip().split("\t"))

# # Initialize arrays to store extracted data
# absoluteT = []
# elapsedT = []
# events = []
# locations = []
        
# # Extract VR data with special handling for 'p' events
# for line in rawVR_data:
#     if len(line) >= 4:  # Ensure line has at least 4 elements
#         absoluteT.append(line[0])
#         elapsedT.append(float(line[1]))
#         events.append(line[2])
        
#         # Extract location based on event type
#         if line[2] == 'p' and len(line) >= 6:  # Position event with enough elements
#             # For 'p' events, get z-coordinate from 6th column
#             locations.append(float(line[5]))
#         else:
#             # For other events, get location from 4th column
#             locations.append(float(line[3]))
            
# # Convert to numpy arrays
# VR_data = {
#     'absoluteT': np.array(absoluteT),
#     'elapsedT': np.array(elapsedT),
#     'event': np.array(events),
#     'location': np.array(locations)
# }

# # Apply minimum threshold to location data
# VR_data['location'][VR_data['location'] < -40] = -40

# Extract VR data
VR_data['absoluteT'] = np.array([line[0] for line in rawVR_data])
VR_data['elapsedT'] = np.array([float(line[1]) for line in rawVR_data])
VR_data['event'] = np.array([line[2] for line in rawVR_data])
VR_data['location'] = np.array([float(line[3]) for line in rawVR_data])

# for any VR_data['location'] that is less than -40, set it to -40
# VR_data['location'][VR_data['location'] < -40] = -40

# Find the index of the first 's' in VR_data['event']
start_index = np.where(VR_data['event'] == 's')[0][0]

# Erase all elements before the start_index in all VR_data
for key in VR_data.keys():
    VR_data[key] = VR_data[key][start_index:]

# # for every element of VR_data, print first value
# for key in VR_data:
#     print(VR_data[key][0])

print("first time value of VR is", VR_data['absoluteT'][0])
print("first time value of 2p is", twoP_data['AbsoluteT'][0] )

**2. Align the 2p data and treadmill behavior data**

In [ ]:
# Define absolute_t0 as the first element of VR_data['absoluteT'] -- with "s" for event type, which is the timestamp for 2p input trigger
VR_absolute_t = np.array([datetime.datetime.strptime(t, '%H.%M.%S.%f') for t in VR_data['absoluteT'][0:]])

# Calculate relative_t (time elapsed from absolute_t0)
VR_relative_t = np.array([(t - VR_absolute_t[0]).total_seconds() for t in VR_absolute_t])

# Add twoP_data['AbsoluteT'][0] to each timedelta object to get vrT
VR_relative_t_timedelta = np.array([datetime.timedelta(seconds=t) for t in VR_relative_t])
Aligned_Abs_vrT = twoP_data['AbsoluteT'][0] + VR_relative_t_timedelta

# Find the closest value in Aligned_Abs_vrT that is greater than twoP_data['AbsoluteT'][-1]
closest_value = Aligned_Abs_vrT[Aligned_Abs_vrT > twoP_data['AbsoluteT'][-1]][0]
closest_index = np.where(Aligned_Abs_vrT == closest_value)[0][0]

# print(closest_value)
# print(closest_index)

new_VR_data = {}
new_VR_data['AbsoluteT'] = np.array(Aligned_Abs_vrT)[:closest_index]
new_VR_data['RelativeT'] = VR_relative_t[:closest_index]
new_VR_data['event'] = VR_data['event'][:closest_index]
new_VR_data['location'] = VR_data['location'][:closest_index]

print(new_VR_data['RelativeT'][-1])
new_VR_data['location'][new_VR_data['location'] < -460] = -460

# Calculate relative time points for VR_data and twoP_data
twop_relativeT = twoP_data['AbsoluteT'] - twoP_data['AbsoluteT'][0]

# Convert to seconds
twop_relativeT = np.array([t.total_seconds() for t in twop_relativeT])
twoP_data['RelativeT'] = twop_relativeT

# print("start of 2p is", twoP_data['RelativeT'][0])
# print("start of VR is", new_VR_data['RelativeT'][0])

# print("end of 2p is", twoP_data['RelativeT'][-1])
# print("end of VR is", new_VR_data['RelativeT'][-1])

# Interpolate the location at twoP_data['RelativeT'] from new_VR_data['location'] at new_VR_data['RelativeT']
interpolated_location = np.interp(twoP_data['RelativeT'], new_VR_data['RelativeT'], new_VR_data['location'])
new_VR_data['interp_location'] = interpolated_location

# # for all other keys in new_VR_data, cut off the last few elements to match the length of interpolated_location
# for key in new_VR_data.keys():
#     new_VR_data[key] = new_VR_data[key][:len(interpolated_location)]

# # Now interpolated_location contains the interpolated location values at twop_relativeT
# print(new_VR_data['location'].shape)
# print(twoP_data['RelativeT'].shape)
# print(interpolated_location.shape)

# Plot the interpolated location
plt.figure(figsize=(20, 5))
plt.plot(twoP_data['RelativeT'], new_VR_data['interp_location']+100, label="Interpolated Location", alpha=0.5)
plt.plot(new_VR_data['RelativeT'], new_VR_data['location'], label="Original Location", alpha=0.5)
plt.legend()
plt.show()

# print(interpolated_location[0:10])
# print(aligned_VR_data['aligned_location'][0:10])

# plt.figure(figsize=(20, 5))
# plt.plot(twoP_data['RelativeT'], twoP_data['dFF'][10, :])
# plt.show()

# plt.figure(figsize=(20, 5))
# plt.plot(twoP_data['RelativeT'], twoP_data['sps'][10, :])
# plt.show()

**3a. Find temporal offset, which yields the best alignment between 2p and behavior data**

In [ ]:
# Define offsets to test (in frames)
offset_frames_list = [1, 2, 3, 4,5, 7]
framerate = 1/rel_time[1]  # Use the same framerate as before

# Dictionary to store results
offset_results = {}

# Loop through offsets
for offset_frames in offset_frames_list:
    print(f"\n\nTesting offset: {offset_frames} frames ({offset_frames/framerate:.2f} seconds)")
    
    # Apply offset to original data
    offset_spike_data = SpikeSmoothing.apply_temporal_offset(twoP_data['sps'], offset_frames)
    
    # Apply smoothing
    smoothed = SpikeSmoothing.smooth_spikes(offset_spike_data, framerate, window_ms=500)
    
    # Process data with trial filtering
    filtered_spks_laps, filtered_location_laps, n_valid_laps = DF.process_data_with_trial_filtering(
        smoothed, 
        new_VR_data['interp_location'],
        min_trial_duration_seconds=5, 
        max_trial_duration_seconds=60,
        framerate=framerate
    )
    
    if n_valid_laps == 0:
        print(f"No valid laps for offset {offset_frames}")
        continue
    
    # single_revolution_VR = 282.415
    # single_revolution_treadmill = 27.8
    # single_lap_VR = 1726.99731 ### = 1146 when VR length was 125 at gain = 1.15 
    
    single_revolution_VR = 282.415
    single_revolution_treadmill = 27.8
    single_lap_VR = 1320.645683 ### = 1146 when VR length was 125 at gain = 1.15 
    single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

    # single_revolution_VR = 282.415
    # single_revolution_treadmill = 27.8
    # single_lap_VR = 1126.0667 ### = 1146 when VR length was 125 at gain = 1.15 
    
    single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR


    # Perform spatial assignment
    spatial_activity, spatial_bins, trial_averaged_activity, bin_centers = SD.spatial_assignment(
        n_valid_laps,
        filtered_spks_laps, 
        filtered_location_laps, 
        single_lap_treadmill
    )
    
    # Apply spatial smoothing
    smoothed_spatial_activity = SpikeSmoothing.spatial_smooth(spatial_activity, window_cm=10)


    # Test for reliability
    reliable_cells, avg_cc, cohens_d, iter_cc, _ = RT.test_cell_reliability(
        smoothed_spatial_activity,
        n_shuffles=100,           
        cc_percentile=90,          
        cohen_threshold=0.8,       
        min_cc_threshold=0.25,      
        min_activity_threshold=0.05, 
    )

    # Store results
    offset_results[offset_frames] = {
        'reliable_cells': reliable_cells,
        'reliable_count': np.sum(reliable_cells),
        'avg_cc': avg_cc,
        'cohens_d': cohens_d,
        'spatial_activity': smoothed_spatial_activity,
        'n_valid_laps': n_valid_laps
    }

    # Print summary
    print(f"Offset {offset_frames}: Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
    print(f"Mean correlation for reliable cells: {np.mean(avg_cc[reliable_cells]):.3f}")
    print(f"Mean Cohen's D for reliable cells: {np.mean(cohens_d[reliable_cells]):.3f}")
    
 # Extract metrics for visualization
valid_offsets = list(offset_results.keys())
reliable_counts = [offset_results[offset]['reliable_count'] for offset in valid_offsets]
avg_cc_means = [np.mean(offset_results[offset]['avg_cc'][offset_results[offset]['reliable_cells']]) 
               if np.sum(offset_results[offset]['reliable_cells']) > 0 else 0
               for offset in valid_offsets]
cohens_d_means = [np.mean(offset_results[offset]['cohens_d'][offset_results[offset]['reliable_cells']])
                 if np.sum(offset_results[offset]['reliable_cells']) > 0 else 0
                 for offset in valid_offsets]

# Create figure
fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

# Plot reliable cell count
axes[0].plot(valid_offsets, reliable_counts, 'o-', color='blue', linewidth=2)
axes[0].set_ylabel('Number of Reliable Cells')
axes[0].set_title('Effect of Temporal Offset on Cell Reliability Metrics')
axes[0].grid(True, alpha=0.3)

# Plot mean correlation coefficient
axes[1].plot(valid_offsets, avg_cc_means, 'o-', color='green', linewidth=2)
axes[1].set_ylabel('Mean Correlation Coefficient')
axes[1].grid(True, alpha=0.3)

# Plot mean Cohen's D
axes[2].plot(valid_offsets, cohens_d_means, 'o-', color='red', linewidth=2)
axes[2].set_xlabel('Offset (frames)')
axes[2].set_ylabel("Mean Cohen's D")
axes[2].grid(True, alpha=0.3)

# Find optimal offset (if results exist)
if len(valid_offsets) > 0:
    # Normalize metrics for weighted average
    norm_reliable = np.array(reliable_counts) / np.max(reliable_counts) if np.max(reliable_counts) > 0 else np.zeros_like(reliable_counts)
    norm_cc = np.array(avg_cc_means) / np.max(avg_cc_means) if np.max(avg_cc_means) > 0 else np.zeros_like(avg_cc_means)
    norm_d = np.array(cohens_d_means) / np.max(cohens_d_means) if np.max(cohens_d_means) > 0 else np.zeros_like(cohens_d_means)
    
    # Weighted sum of normalized metrics
    combined_metric = (0.2 * norm_reliable + 0.4 * norm_cc + 0.4 * norm_d)
    # combined_metric = (norm_reliable + norm_cc + norm_d) / 3
    best_idx = np.argmax(combined_metric)
    best_offset = valid_offsets[best_idx]
    
    # Add vertical line at optimal offset
    for ax in axes:
        ax.axvline(x=best_offset, color='black', linestyle='--', alpha=0.7)
        ax.text(best_offset, ax.get_ylim()[1]*0.95, f'Optimal: {best_offset}', 
               horizontalalignment='center', verticalalignment='top')
    
    print(f"\nBest offset: {best_offset} frames ({best_offset/framerate:.2f} seconds)")
    print(f"- Reliable cells: {offset_results[best_offset]['reliable_count']}")
    print(f"- Mean correlation: {np.mean(offset_results[best_offset]['avg_cc'][offset_results[best_offset]['reliable_cells']]):.3f}")
    print(f"- Mean Cohen's D: {np.mean(offset_results[best_offset]['cohens_d'][offset_results[best_offset]['reliable_cells']]):.3f}")

plt.tight_layout()
plt.show()   

**3b. Smooth the deconvolved traces using a 500 ms Gaussian window**

In [ ]:
print(best_offset)
offset_spike_data = SpikeSmoothing.apply_temporal_offset(twoP_data['sps'], best_offset)
smoothed = SpikeSmoothing.smooth_spikes(offset_spike_data, framerate, window_ms=500)
# smoothed = SpikeSmoothing.smooth_spikes(twoP_data['sps'], framerate, window_ms=500)
twoP_data['smoothed_spks'] = smoothed
# SpikeSmoothing.plot_comparison(twoP_data['s2p_spks'], smoothed, framerate)

**3c. Remove inactive data points**

In [ ]:
### 1) any time points where the treadmill behavior data is not available (i.e. when the animal is not moving)
### 2) any trial that is taking less than 5 seconds or more than 30 seconds to complete
min_trial_duration_seconds = 5
max_trial_duration_seconds = 60

filtered_spks_laps, filtered_location_laps, n_valid_laps = DF.process_data_with_trial_filtering(
    twoP_data['smoothed_spks'], 
    new_VR_data['interp_location'],
    min_trial_duration_seconds = min_trial_duration_seconds, 
    max_trial_duration_seconds = max_trial_duration_seconds,
    framerate=framerate
)

**4. Discretize spatial data (1cm/bin) and assign 2p data to a corresponding bin**

In [ ]:
# # when VR length was 300 at gain = 1.15 - 25/03/20 
single_revolution_VR = 282.415
single_revolution_treadmill = 27.8
# single_lap_VR = 1726.99731 ### = 1146 when VR length was 125 at gain = 1.15 
single_lap_VR = 1320.645683 ### = 1146 when VR length was 125 at gain = 1.15 
single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

# # when VR length was 125 at gain = 1.15 
# single_revolution_VR = 285.9317
# single_revolution_treadmill = 27.8
# single_lap_VR = 1146 ### = 1146 when VR length was 125 at gain = 1.15 
# single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

# Then perform spatial assignment on the filtered data
spatial_activity, spatial_bins, trial_averaged_activity, bin_centers = SD.spatial_assignment(
    n_valid_laps,
    filtered_spks_laps, 
    filtered_location_laps, 
    single_lap_treadmill
)

window_cm = 5
smoothed_spatial_activity = SpikeSmoothing.spatial_smooth(spatial_activity, window_cm=window_cm)
normalized_spatial_activity = RT.normalize_spatial_activity(smoothed_spatial_activity)

**5. Test for reliability for individual cells (uses Pearson CC and cohen’s D)**

In [ ]:
# Run the analysis
reliable_cells, avg_cc, cohens_d, iter_cc, _ = RT.test_cell_reliability(
    smoothed_spatial_activity,
    n_shuffles=100,           # Use 1000+ for final analysis
    cc_percentile=90,          # 90th percentile threshold for CC
    cohen_threshold=0.8,       # Medium-large effect size
    min_cc_threshold=0.3,      # Minimum correlation required
    min_activity_threshold=0.1 # Minimum activity level (relative to max)
)

# Print summary
print(f"Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
print(f"Mean correlation for reliable cells: {np.mean(avg_cc[reliable_cells]):.3f}")
print(f"Mean Cohen's D for reliable cells: {np.mean(cohens_d[reliable_cells]):.3f}")

In [ ]:
def evaluate_pattern_similarity(spatial_activity, min_pattern_corr=0.4, peak_distance_threshold=5):
    """
    Evaluates both correlation and pattern similarity between odd and even trials.
    
    Parameters:
    -----------
    spatial_activity : numpy.ndarray
        Activity matrix (n_cells x n_trials x n_spatial_bins)
    min_pattern_corr : float
        Minimum correlation threshold for pattern similarity
    peak_distance_threshold : int
        Maximum allowed distance between peaks in odd and even trials (in bins)
        
    Returns:
    --------
    pattern_reliable : numpy.ndarray
        Boolean array of cells with consistent patterns
    odd_even_corr : numpy.ndarray
        Correlation between odd and even trials for each cell
    peak_distances : numpy.ndarray
        Distance between peaks in odd and even trials for each cell
    """
    n_cells, n_trials, n_bins = spatial_activity.shape
    
    # Initialize output arrays
    pattern_reliable = np.zeros(n_cells, dtype=bool)
    odd_even_corr = np.zeros(n_cells)
    peak_distances = np.zeros(n_cells)
    
    # Separate odd and even trials
    odd_trials = np.arange(0, n_trials, 2)
    even_trials = np.arange(1, n_trials, 2)
    
    for cell in range(n_cells):
        # Calculate mean activity for odd and even trials
        odd_mean = np.mean(spatial_activity[cell, odd_trials], axis=0)
        even_mean = np.mean(spatial_activity[cell, even_trials], axis=0)
        
        # Calculate correlation between odd and even mean activity patterns
        odd_even_corr[cell] = np.corrcoef(odd_mean, even_mean)[0, 1]
        
        # Find peaks in odd and even trials
        odd_peak = np.argmax(odd_mean)
        even_peak = np.argmax(even_mean)
        
        # Calculate shortest distance between peaks (accounting for circular track)
        raw_distance = abs(odd_peak - even_peak)
        circular_distance = min(raw_distance, n_bins - raw_distance)
        peak_distances[cell] = circular_distance
        
        # Check if patterns are similar (high correlation and consistent peak location)
        if (odd_even_corr[cell] >= min_pattern_corr and 
            peak_distances[cell] <= peak_distance_threshold):
            pattern_reliable[cell] = True
    
    return pattern_reliable, odd_even_corr, peak_distances

def combined_reliability_test(spatial_activity, n_shuffles=1000, 
                             cc_percentile=95, cohen_threshold=0.5,
                             min_cc_threshold=0.2, min_activity_threshold=0.1,
                             min_pattern_corr=0.4, peak_distance_threshold=5):
    """
    Combined reliability test checking both trial-to-trial reliability and odd-even pattern similarity
    """
    # Run original reliability test
    reliable_cells, avg_cc, cohens_d, iter_cc, norm_activity = RT.test_cell_reliability(
        spatial_activity, n_shuffles, cc_percentile, cohen_threshold,
        min_cc_threshold, min_activity_threshold
    )
    
    # Run pattern similarity test
    pattern_reliable, odd_even_corr, peak_distances = evaluate_pattern_similarity(
        spatial_activity, min_pattern_corr, peak_distance_threshold
    )
    
    # Combine results (cells must pass both tests)
    combined_reliable = reliable_cells & pattern_reliable
    
    return (combined_reliable, reliable_cells, pattern_reliable, 
            avg_cc, cohens_d, odd_even_corr, peak_distances, norm_activity)
    
    
# Run the analysis
combined_reliable, reliable_cells, _, avg_cc, cohens_d, _, _, _ = combined_reliability_test(
    smoothed_spatial_activity,
    n_shuffles=100,           # Use 1000+ for final analysis
    cc_percentile=90,          # 90th percentile threshold for CC
    cohen_threshold=0.8,       # Medium-large effect size
    min_cc_threshold=0.3,      # Minimum correlation required
    min_activity_threshold=0.1, # Minimum activity level (relative to max)
    min_pattern_corr=0.4, 
    peak_distance_threshold=5
)

print(f"Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
print(f"Found {np.sum(combined_reliable)} reliable cells out of {len(combined_reliable)}")


In [ ]:
# # Option 1.1 Plot the average activity oin a grid layout
# reliable_avg = {}

# for i in np.where(reliable_cells)[0]:
#     reliable_avg[i] = np.mean(spatial_activity[i, :, :], axis=0)

# # plot average activity of first 25 reliable cells in a 5 x 5 subplots
# fig, axs = plt.subplots(5, 5, figsize=(20, 20))
# reliable_indices = list(reliable_avg.keys())[:25]
# for idx, i in enumerate(reliable_indices):
#     ax = axs[idx // 5, idx % 5]
#     ax.plot(bin_centers, reliable_avg[i])
#     ax.set_title(f'Cell {i}')
#     ax.set_xlabel('Position')
#     ax.set_ylabel('Activity')

# # set a title over all plots
# fig.suptitle('Average spikes of example reliable cells', fontsize=16)
# plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust the rect parameter to add space at the top
# plt.show()

# # Option 1.2: Plot heatmaps in a grid layout (eg. 20 cells in a 5×4 grid)
# fig1 = RT.plot_reliable_cells_grid(
#     normalized_spatial_activity,
#     reliable_cells,
#     max_cells=20,                # Show up to 20 reliable cells
#     avg_cc=avg_cc,               # Optional correlation coefficients
#     cohen_d=cohens_d,            # Optional Cohen's D values
#     normalize=False,              # Apply normalization
#     n_rows=5,                    # 5 rows
#     n_cols=4                     # 4 columns
# )
# plt.suptitle('Spikes of reliable cells', fontsize=15)
# plt.tight_layout(rect=[0, 0, 1, 0.985])  # Adjust the rect parameter to add space at the top

# plt.show()

# Option 2: Plot with both heatmap and trial-averaged activity
fig2 = RT.plot_reliable_cells_side_by_side(
    normalized_spatial_activity,
    reliable_cells,
    # combined_reliable,
    # max_cells=50,              # Show up to 10 reliable cells
    max_cells=np.sum(reliable_cells),                # Show up to 10 reliable cells
    avg_cc=avg_cc,               # Optional correlation coefficients
    cohen_d=cohens_d,            # Optional Cohen's D values
    normalize=False               # Apply normalization
)
plt.suptitle('Spikes of reliable cells', fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.985])  # Adjust the rect parameter to add space at the top
plt.show()

# # Option 3: Plot waterfall plots
# RT.plot_reliable_cells_waterfall(normalized_spatial_activity, reliable_cells, max_cells=np.sum(reliable_cells))
# plt.show()

**6. Response Plot - plotting activity of all responsive cells (cross validation – split trials in half)**

In [ ]:
# Create the response plot with a larger figure size
fig1, sorted_indices = RV.create_response_plot(normalized_spatial_activity, reliable_cells, clim=(0, 1))  # Optional: manually set contrast limits for stronger effect

# # Add red and green vertical lines in both subplots of fig1
# for ax in fig1.axes:
#     # Add red vertical lines at 50, 90, 130 and label as landmark A
#     for i in range(50, 151, 40):
#         ax.axvline(x=i, color='red', linestyle='--', alpha=0.5)
#         ax.axhline(y=245, color='red', linestyle='--', alpha=0.3)
#         ax.axhline(y=325, color='red', linestyle='--', alpha=0.3)
#         ax.fill_betweenx([245, 325], 50, 70, color='red', alpha=0.1)
        
#         ax.axhline(y=335, color='red', linestyle='--', alpha=0.3)
#         ax.axhline(y=350, color='red', linestyle='--', alpha=0.3)
#         ax.fill_betweenx([335, 350], 90, 110, color='red', alpha=0.1)
        
#         ax.axhline(y=370, color='red', linestyle='--', alpha=0.3)
#         ax.axhline(y=420, color='red', linestyle='--', alpha=0.3)
#         ax.fill_betweenx([370, 420], 130, 150, color='red', alpha=0.1)
        
#         ax.text(i, 0.95, 'A', color='red', fontsize=10, ha='center', va='top', transform=ax.get_xaxis_transform())
    
#     # Add green vertical lines at 70, 110, 150 and label as landmark B
#     for i in range(70, 151, 40):
#         ax.axvline(x=i, color='green', linestyle='--', alpha=0.5)
#         ax.text(i, 0.95, 'B', color='green', fontsize=10, ha='center', va='top', transform=ax.get_xaxis_transform())

plt.show()

# # add red vertical lines at 50, 90,130 and label as landmark A, green vertical lines at 70, 110, 150 and label as landmark B
# for i in range(50, 151, 40):
#     plt.axvline(x=i, color='red', linestyle='--', alpha=0.5)
#     plt.text(i, 0.95, 'A', color='red', fontsize=10, ha='center', va='top')
# for i in range(70, 151, 40):
#     plt.axvline(x=i, color='green', linestyle='--', alpha=0.5)
#     plt.text(i, 0.95, 'B', color='green', fontsize=10, ha='center', va='top')
    
    
# # # For waterfall visualization (alternative approach):
# # fig2 = RV.create_waterfall_plot(normalized_spatial_activity, reliable_cells)

# plt.show()     

In [ ]:
def save_preprocessed_data_pickle(save_path=None):
    """
    Save preprocessed neuroimaging data to a pickle file.
    """
    import pickle
    import os
    from tkinter import Tk, filedialog, simpledialog
    
    # Data to save - create a dictionary with all variables
    data_to_save = {
        # Metadata
        'filepath': filepath if 'filepath' in globals() else None,
        'twoP_filename': twoP_filename if 'twoP_filename' in globals() else None,
        'VRlog_filename': VRlog_filename if 'VRlog_filename' in globals() else None,
        'VRlog_path': VRlog_path if 'VRlog_path' in globals() else None,
        
        # Original data
        'twoP_data': twoP_data if 'twoP_data' in globals() else None,
        'new_VR_data': new_VR_data if 'new_VR_data' in globals() else None,
        'filtered_spks_laps': filtered_spks_laps if 'filtered_spks_laps' in globals() else None,
        'filtered_location_laps': filtered_location_laps if 'filtered_location_laps' in globals() else None,
        
        # Parameters
        'n_valid_laps': n_valid_laps if 'n_valid_laps' in globals() else None,
        'framerate': framerate if 'framerate' in globals() else None,
        'best_offset': best_offset if 'best_offset' in globals() else None,
        'min_trial_duration_seconds': min_trial_duration_seconds if 'min_trial_duration_seconds' in globals() else None,
        'max_trial_duration_seconds': max_trial_duration_seconds if 'max_trial_duration_seconds' in globals() else None,
        
        # Neural activity data
        'spatial_activity': spatial_activity if 'spatial_activity' in globals() else None,
        'trial_averaged_activity': trial_averaged_activity if 'trial_averaged_activity' in globals() else None,
        'smoothed_spatial_activity': smoothed_spatial_activity if 'smoothed_spatial_activity' in globals() else None,
        'normalized_spatial_activity': normalized_spatial_activity if 'normalized_spatial_activity' in globals() else None,
        
        # Spatial information
        'single_lap_VR': single_lap_VR if 'single_lap_VR' in globals() else None,
        'single_lap_treadmill': single_lap_treadmill if 'single_lap_treadmill' in globals() else None,
        'window_cm': window_cm if 'window_cm' in globals() else None,
        'bin_centers': bin_centers if 'bin_centers' in globals() else None,
        'spatial_bins': spatial_bins if 'spatial_bins' in globals() else None,
        
        # Cell information
        'combined_reliable': combined_reliable if 'combined_reliable' in globals() else None,
        'reliable_cells': reliable_cells if 'reliable_cells' in globals() else None,
        'avg_cc': avg_cc if 'avg_cc' in globals() else None,
        'cohens_d': cohens_d if 'cohens_d' in globals() else None,
        'sorted_indices': sorted_indices if 'sorted_indices' in globals() else None,
        
        # image data
        'im': im if 'im' in globals() else None
    }
    
    # Remove None values (variables that don't exist)
    data_to_save = {k: v for k, v in data_to_save.items() if v is not None}
    
    # If no save path provided, ask user for one
    if save_path is None:
        root = Tk()
        root.withdraw()  # Hide the main window
        
        # Ask for a filename
        default_filename = f"{twoP_filename}_preprocessed.pkl" if 'twoP_filename' in globals() else "data_preprocessed.pkl"
        save_filename = simpledialog.askstring("Save File", 
                                              "Enter filename for saving preprocessed data:",
                                              initialvalue=default_filename)
        
        if save_filename:
            if not save_filename.endswith('.pkl'):
                save_filename += '.pkl'
            
            # Ask for directory to save
            save_dir = filedialog.askdirectory(title="Select directory to save data")
            
            if save_dir:
                save_path = os.path.join(save_dir, save_filename)
            else:
                print("No directory selected. Data not saved.")
                return
        else:
            print("No filename provided. Data not saved.")
            return
    
    # Save the data
    try:
        with open(save_path, 'wb') as f:
            pickle.dump(data_to_save, f)
        
        print(f"All data saved to: {save_path}")
        print(f"Saved {len(data_to_save)} variables")
        return save_path
    
    except Exception as e:
        print(f"Error saving data: {e}")
        import traceback
        traceback.print_exc()
        return None

save_preprocessed_data_pickle()